## 0. import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. 데이터 준비

In [3]:
RANDOM_STATE = 203

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

def to_grade(q):
    if q <= 4:
        return 'low'
    elif q <= 6:
        return 'medium'
    else:
        return 'high'

df['grade'] = df['quality'].apply(to_grade)

X = df.drop(columns=['quality', 'grade'])
y = df['grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## 2. Voting Classifier

In [ ]:
lr_clf = LogisticRegression(max_iter=1000)
knn_clf = KNeighborsClassifier(n_neighbors=8)

vo_clf = VotingClassifier(
    estimators=[('LR', lr_clf), ('KNN', knn_clf)], voting='soft'
)
vo_clf.fit(X_train, y_train)
vo_pred = vo_clf.predict(X_test)
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test, vo_pred)))

# 개별 모델 성능과 비교
for clf in [lr_clf, knn_clf]:
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print('{0} 정확도: {1:.4f}'.format(
        clf.__class__.__name__, accuracy_score(y_test, pred)))

## 3. 단일 트리 vs 랜덤포레스트 비교

In [ ]:
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)
print('단일 트리 정확도: {0:.4f}'.format(accuracy_score(y_test, dt_pred)))

rf_clf = RandomForestClassifier(
    n_estimators=100, max_depth=4, random_state=RANDOM_STATE, n_jobs=-1
)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
print('랜덤포레스트 정확도: {0:.4f}'.format(accuracy_score(y_test, rf_pred)))

print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

## 4. GridSearchCV로 랜덤포레스트 튜닝

In [ ]:
params = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8, 16],
    'min_samples_leaf': [1, 5, 10]
}

rf_clf_grid = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
grid_cv = GridSearchCV(rf_clf_grid, param_grid=params, cv=5, n_jobs=-1)
grid_cv.fit(X_train, y_train)

print('최적 하이퍼파라미터:\n', grid_cv.best_params_)
print('최고 교차검증 정확도: {0:.4f}'.format(grid_cv.best_score_))

# 최적 파라미터로 재학습 후 테스트 세트 평가
best_rf = RandomForestClassifier(
    **grid_cv.best_params_, random_state=RANDOM_STATE, n_jobs=-1
)
best_rf.fit(X_train, y_train)
best_pred = best_rf.predict(X_test)
print('최종 테스트 정확도: {0:.4f}'.format(accuracy_score(y_test, best_pred)))


## 5. 특성 중요도 시각화

In [ ]:
ftr_importances_values = best_rf.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index=X_train.columns)
ftr_top = ftr_importances.sort_values(ascending=False)

plt.figure(figsize=(8, 6))
plt.title('Random Forest Feature Importances (Wine Quality)')
sns.barplot(x=ftr_top, y=ftr_top.index)
plt.show()

## 6. 성능 비교 요약표

In [ ]:
summary = pd.DataFrame({
    'model': ['DecisionTree', 'RandomForest(default)', 'RandomForest(tuned)', 'VotingClassifier'],
    'test_accuracy': [
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, best_pred),
        accuracy_score(y_test, vo_pred),
    ]
})
print(summary)